# Fake News Detection Using Machine Learning

**Aditya Mittal**


In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn kagglehub


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


In [ ]:
dataset_path = Path(
    kagglehub.dataset_download(
        "clmentbisaillon/fake-and-real-news-dataset"
    )
)

fake_file = list(dataset_path.rglob("Fake.csv"))[0]
true_file = list(dataset_path.rglob("True.csv"))[0]

print(fake_file)
print(true_file)


In [ ]:
fake_df = pd.read_csv(fake_file)
true_df = pd.read_csv(true_file)

fake_df.head()


In [ ]:
true_df.head()


In [ ]:
fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat(
    [fake_df, true_df],
    ignore_index=True
)

df = df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

df.head()


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
print(df.isna().sum())
print(df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().copy()

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df["content"] = (
    df["title"].astype(str)
    + " "
    + df["text"].astype(str)
)

df = df[
    df["content"].str.strip() != ""
].copy()


In [ ]:
label_counts = df["label"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(
    ["Fake", "Real"],
    [
        label_counts.get(0, 0),
        label_counts.get(1, 0)
    ]
)
plt.title("News Class Distribution")
plt.ylabel("Number of Articles")
plt.show()


In [ ]:
df["text_length"] = df["content"].str.len()

plt.figure(figsize=(8, 5))
plt.hist(
    df[df["label"] == 0]["text_length"],
    bins=40,
    alpha=0.6,
    label="Fake"
)
plt.hist(
    df[df["label"] == 1]["text_length"],
    bins=40,
    alpha=0.6,
    label="Real"
)
plt.title("Article Length Distribution")
plt.xlabel("Text Length")
plt.ylabel("Articles")
plt.legend()
plt.show()


In [ ]:
X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    min_df=2,
    max_features=20000,
    ngram_range=(1, 2)
)

X_train_vectorized = vectorizer.fit_transform(
    X_train
)

X_test_vectorized = vectorizer.transform(
    X_test
)

print(X_train_vectorized.shape)
print(X_test_vectorized.shape)


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=25,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=30,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

trained_models = {}
results = []

for model_name, model in models.items():

    model.fit(
        X_train_vectorized,
        y_train
    )

    predictions = model.predict(
        X_test_vectorized
    )

    results.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(
                y_test,
                predictions
            ),
            "Precision": precision_score(
                y_test,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_test,
                predictions,
                zero_division=0
            ),
            "F1 Score": f1_score(
                y_test,
                predictions,
                zero_division=0
            )
        }
    )

    trained_models[model_name] = model

results_df = pd.DataFrame(
    results
).sort_values(
    by="F1 Score",
    ascending=False
)

results_df.round(4)


In [ ]:
graph_data = results_df.sort_values(
    "F1 Score"
)

plt.figure(figsize=(8, 5))
plt.barh(
    graph_data["Model"],
    graph_data["F1 Score"]
)
plt.title("Model Comparison")
plt.xlabel("F1 Score")
plt.xlim(0, 1)
plt.show()


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

final_predictions = best_model.predict(
    X_test_vectorized
)

print(best_model_name)
print(
    classification_report(
        y_test,
        final_predictions,
        target_names=[
            "Fake",
            "Real"
        ],
        zero_division=0
    )
)


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_predictions,
    display_labels=[
        "Fake",
        "Real"
    ]
)

plt.title(best_model_name)
plt.show()


In [ ]:
logistic_model = trained_models[
    "Logistic Regression"
]

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

coefficients = logistic_model.coef_[0]

fake_indices = np.argsort(coefficients)[:20]
real_indices = np.argsort(coefficients)[-20:]

fake_words = pd.DataFrame(
    {
        "Word": feature_names[fake_indices],
        "Weight": coefficients[fake_indices]
    }
)

real_words = pd.DataFrame(
    {
        "Word": feature_names[real_indices],
        "Weight": coefficients[real_indices]
    }
)

fake_words


In [ ]:
real_words


In [ ]:
sample_news = [
    "Government announces a new education policy after official cabinet approval."
]

sample_vector = vectorizer.transform(
    sample_news
)

sample_prediction = best_model.predict(
    sample_vector
)[0]

if sample_prediction == 1:
    print("Real News")
else:
    print("Fake News")
